# Step 6: Multiple Tools

        _Instructor Solution Notebook_  
        Estimated time: **60 minutes**  
        Difficulty: **Intermediate**

        ## Learning Objectives
        - [ ] Give one agent several tools with distinct purposes.
- [ ] Observe tool choice across different user requests.
- [ ] Think about tool overlap and ambiguity.
- [ ] Prepare for async and API-backed tools in the next step.

        ## Prerequisites
        - Step 5 completed.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Summary & Key Takeaways
8. Additional Resources


## Setup & Imports

The next cell adds the repository root to `sys.path` and imports the shared course helpers.
Most notebooks use the same bootstrap so the lesson content can stay focused on the concept,
not on path setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Multiple tools introduce selection pressure. The model now needs good descriptions and enough examples to decide which function helps most.

Expected output and validation notes:

Expected output snapshot:

- math prompt routes to `calculate`
- conversion prompt routes to `convert_c_to_f`
- time prompt routes to `get_current_time`


## Part 2: Core Implementation

### Define a small toolbelt


In [ ]:
from agent_framework import tool
from datetime import datetime

@tool
def calculate(expression: str) -> str:
    """Evaluate a simple arithmetic expression safely for notebook demos."""
    allowed = {"__builtins__": {}}
    return str(eval(expression, allowed, {}))

@tool
def convert_c_to_f(celsius: float) -> float:
    """Convert Celsius to Fahrenheit."""
    return round((celsius * 9 / 5) + 32, 2)

@tool
def get_current_time() -> str:
    """Return the current local timestamp."""
    return datetime.now().isoformat(timespec="seconds")

multi_tool_agent = create_agent(
    name="MultiToolAgent",
    instructions="Use the most relevant tool for math, time, and temperature requests.",
    tools=[calculate, convert_c_to_f, get_current_time],
)


## Part 2: Core Implementation

### Exercise the tool selection


In [ ]:
prompts = [
    "What is 15 + 27 * 3?",
    "Convert 30C to Fahrenheit.",
    "What time is it right now?",
]

for prompt in prompts:
    print_banner(prompt)
    reply = await multi_tool_agent.run(prompt)
    print(reply.text)


## Part 3: Hands-On Exercises

Add the Step 5 weather tool so the same agent can answer weather and arithmetic questions in one session.

This mirrored notebook uses completed exercise code so instructors can demonstrate the target state.


In [ ]:
from agent_framework import tool

@tool
def get_weather(city: str) -> str:
    """Return a deterministic weather sample for a city."""
    samples = {"Paris": "Sunny, 22C", "Seattle": "Cloudy, 14C"}
    return samples.get(city, f"No sample weather for {city}")

blended_agent = create_agent(
    name="BlendedAgent",
    instructions="Use tools whenever a question is factual or computational.",
    tools=[calculate, convert_c_to_f, get_current_time, get_weather],
)
for prompt in ["What's the weather in Seattle?", "What is 3 * (8 + 4)?"]:
    print_banner(prompt)
    result = await blended_agent.run(prompt)
    print(result.text)


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
from agent_framework import tool

@tool
def get_weather(city: str) -> str:
    """Return a deterministic weather sample for a city."""
    samples = {"Paris": "Sunny, 22C", "Seattle": "Cloudy, 14C"}
    return samples.get(city, f"No sample weather for {city}")

blended_agent = create_agent(
    name="BlendedAgent",
    instructions="Use tools whenever a question is factual or computational.",
    tools=[calculate, convert_c_to_f, get_current_time, get_weather],
)
for prompt in ["What's the weather in Seattle?", "What is 3 * (8 + 4)?"]:
    print_banner(prompt)
    result = await blended_agent.run(prompt)
    print(result.text)


## Part 5: Best Practices & Tips

        - Make tool names distinct so the model has clear selection cues.
- Avoid two tools that answer nearly the same request in different ways.
- Test tools individually before blending them into one agent.


## Summary & Key Takeaways

You now have a coordinated multi-tool agent and a framework for debugging tool selection.


## Additional Resources

        - `Step 5 notebook`
- `helpers/llm.py`
